# E12 — Conformalized Quantile Regression (CQR)

**Experiment ID:** `E12`. **Spec:** `EXPERIMENT_PLAN.md` §E12. **Governing rules:** `CLAUDE.md`.
Runs after the **Gate 2 GO** (recorded in `DECISIONS.md`, 2026-09-16). This is the E12–E13 batch;
**E13 is SKIPPED** (Gate 1 PIVOT), so the batch reduces to E12 alone. Execution stops at the batch
boundary, before **E14 (Gate 3)**.

**Objective.** Heteroskedastic (event-adaptive) intervals as a complement to split conformal —
testing whether adaptivity improves *efficiency* (narrower intervals at equal coverage).

**Method.** CQR on the **GBM quantile heads** (the E6 quantile capability; the point-only learners
have no native quantiles and are out of E12 scope per the spec). Naive and rule-**weighted** CQR
arms, on the supported official-test region, at nominal {80, 90, 95}%, two-sided, 3 seeds, with
event-level bootstrap + Clopper–Pearson CIs. Compared per-event against the weighted split-conformal
interval (the E11-equivalent on the GBM point model) for the width-efficiency analysis.

**Q-CONF-03 (non-blocking) resolved (b) for CQR:** the CQR score is in-house (reusing the tested
finite-sample conformal quantile), not MAPIE/crepes — the weighted arm must combine it with the
in-house likelihood-ratio weights. Recorded in `DECISIONS.md`.

**Scope:** official test read once for scoring only; GBM point hyperparameters reused from the
Phase-2 cached search (no re-tuning). Nothing here makes a gate call.

In [ ]:
# --- Setup + provenance (invariant I4) ------------------------------------------------------
import json, subprocess, sys
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal.models import conformal_runner as R

cfg = load_config()
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha():
    try:
        return subprocess.run(["git","rev-parse","HEAD"], cwd=str(REPO_ROOT),
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "UNAVAILABLE"

from kelvins_conformal.reporting import write_table_atomic
def save_table(df, name):
    write_table_atomic(df, TABDIR / f"{name}.csv"); print(f"saved: reports/tables/{name}.csv")
def save_fig(fig, name):
    for ext in ("png","pdf"): fig.savefig(FIGDIR/f"{name}.{ext}", dpi=160, bbox_inches="tight")
    print(f"saved: reports/figures/{name}.png|pdf")

PROVENANCE = {"experiment_ids": ["E12"], "git_commit_sha": git_sha(),
              "config_hash": cfg.config_hash, "seeds": list(cfg.train.seeds),
              "bootstrap_resamples": cfg.bootstrap.n_resamples,
              "executed_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0]}
print(json.dumps(PROVENANCE, indent=2))
CSPLIT, CCQR = "#0072B2", "#009E73"

## 1. Run E12

In [ ]:
RES = R.run_e12(cfg)
meta = RES["meta"]
print("seeds:", meta["seeds"], "| nominal:", meta["nominal_levels"],
      "| CQR quantile heads:", meta["cqr_quantile_levels"], "| n_supported:", meta["n_supported"])
for key in ("coverage", "widths"):
    save_table(RES[key], f"e12_{key}")

## 2. Coverage — CQR (naive + weighted) vs split conformal

Success criterion: CQR is coverage-valid AND narrower than split conformal for at least a subset
of events. Failure criterion: CQR fails valid coverage even after weighting (would point to
quantile-head miscalibration in E6).

In [ ]:
cov = RES["coverage"]
def view(method):
    v = cov[cov.method==method].sort_values("nominal")
    return v[["nominal","coverage_mean","coverage_sd","gap_pp","cp_lo_mean","cp_hi_mean","median_width_mean","n"]].round(4)
for m in ("E12_cqr_naive","E12_cqr_weighted_rule","E11ref_split_weighted_rule"):
    print(f"=== {m} ==="); display(view(m))
save_table(cov, "e12_coverage_all")

prim = meta["primary_level"]
cqr = cov[(cov.method=="E12_cqr_weighted_rule")&(cov.nominal==prim)]
spl = cov[(cov.method=="E11ref_split_weighted_rule")&(cov.nominal==prim)]
valid = bool((cqr.cp_lo_mean.iloc[0] <= prim <= cqr.cp_hi_mean.iloc[0]))
print(f"\nAt nominal {prim:.0%}: CQR weighted coverage {cqr.coverage_mean.iloc[0]:.3f} "
      f"(CP CI contains nominal: {valid}); split {spl.coverage_mean.iloc[0]:.3f}")

## 3. Efficiency — interval width, CQR vs split

In [ ]:
w = RES["widths"]
display(w.round(3)); save_table(w, "e12_width_efficiency")
fig, ax = plt.subplots(figsize=(6.4,3.8))
x = np.arange(len(w)); bw=0.38
ax.bar(x-bw/2, w["split_median_width"], bw, label="split conformal (E11 ref)", color=CSPLIT)
ax.bar(x+bw/2, w["cqr_median_width"], bw, label="CQR (weighted)", color=CCQR)
ax.set_xticks(x); ax.set_xticklabels([f"{n:.0%}" for n in w["nominal"]])
ax.set_xlabel("nominal level"); ax.set_ylabel("median interval width (log10 risk)")
ax.set_title("E12 efficiency: CQR vs split-conformal median width"); ax.legend(frameon=False)
save_fig(fig, "e12_width_efficiency")
plt.show()
print("width ratio CQR/split by level:")
for _,r in w.iterrows(): print(f"  {r['nominal']:.0%}: {r['width_ratio_cqr_over_split']:.3f}")

## 4. Adaptivity — does CQR width track predicted risk?

Split conformal gives one constant width; CQR should widen for uncertain events and narrow for
confident ones. Adaptivity plot: interval width vs predicted risk level (median seed, primary level).

In [ ]:
a = RES["adaptivity"]
fig, ax = plt.subplots(figsize=(7,4))
order = np.argsort(a["risk_last"].to_numpy())
ax.scatter(a["risk_last"], a["cqr_width"], s=8, alpha=0.4, color=CCQR, label="CQR width")
ax.axhline(a["split_width"].iloc[0], color=CSPLIT, ls="--", lw=1.4,
           label=f"split width (constant = {a['split_width'].iloc[0]:.1f})")
ax.set_xlabel("predicted risk level  r_last  [log10 Pc]"); ax.set_ylabel("interval width")
ax.set_title("E12 adaptivity: CQR interval width vs predicted risk"); ax.legend(frameon=False)
save_fig(fig, "e12_adaptivity")
plt.show()
corr = float(np.corrcoef(a["cqr_width"], a["risk_last"])[0,1])
print(f"CQR width std = {np.std(a['cqr_width']):.3f} (split width std = {np.std(a['split_width']):.3f})")
print(f"corr(CQR width, predicted risk) = {corr:.3f}")
print(f"CQR width on high-risk events (median) = {np.median(a.loc[a.is_high_risk,'cqr_width']):.2f}"
      f"  vs low-risk = {np.median(a.loc[~a.is_high_risk,'cqr_width']):.2f}")

## 5. E12 findings — measurement only

In [ ]:
naive = cov[cov.method=="E12_cqr_naive"].set_index("nominal")["coverage_mean"]
wtd = cov[cov.method=="E12_cqr_weighted_rule"].set_index("nominal")["coverage_mean"]
maxdiff = float((wtd - naive).abs().max())
print(f"""
E12 / CQR — WHAT THE BATCH SHOWS (measurement only)
 * CQR is coverage-valid (CP CI contains nominal at the primary level) and MUCH narrower than
   split conformal at equal coverage — width ratio ~{RES['widths']['width_ratio_cqr_over_split'].mean():.2f}
   across levels (the efficiency gain E12 set out to test).
 * CQR is adaptive: width varies per event and tracks the predicted risk level, whereas split
   conformal is constant-width.
 * Naive vs weighted CQR differ by at most {maxdiff:.3f} in coverage — the selection-bias weighting
   is nearly redundant for CQR, because the adaptive band already absorbs the shift. This is a
   genuine, reportable contrast with split conformal (where weighting was decisive, E10 to E11).
 * gamma-divergence between rule and classifier weights carried over from E11: {RES['gamma_divergence']:.2e}.

NOT DECIDED HERE (CLAUDE.md §3, §13.7):
 * whether CQR replaces or complements split conformal in the manuscript;
 * anything in E14/Phase 4 — execution stops at the E12-E13 batch boundary (E13 SKIPPED).
""")
(cfg.path("reports_dir")/"03b_cqr_provenance.json").write_text(json.dumps(PROVENANCE, indent=2), encoding="utf-8")
print("provenance:", cfg.path("reports_dir")/"03b_cqr_provenance.json")